<a href="https://colab.research.google.com/github/shiv2899/tiny-gpt-from-scratch/blob/main/tiny_gpt_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
print("Torch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Torch: 2.14.1+cpu
GPU Available: False
GPU Name: No GPU


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

base = "/content/drive/MyDrive/tiny-gpt-from-scratch"
os.makedirs(f"{base}/data", exist_ok=True)
os.makedirs(f"{base}/checkpoints", exist_ok=True)
print("Folder ban gaya:", base)

!wget -q -O {base}/data/input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

!ls -lh {base}/data/input.txt
!head -c 500 {base}/data/input.txt

Folder ban gaya: /content/drive/MyDrive/tiny-gpt-from-scratch
-rw------- 1 root root 1.1M Nov 18 08:03 /content/drive/MyDrive/tiny-gpt-from-scratch/data/input.txt
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. Data Load ---
with open('/content/drive/MyDrive/tiny-gpt-from-scratch/data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Total characters:", len(text))
print(text[:200])

# --- 2. Vocab banao ---
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocab size:", vocab_size)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print("Train:", len(train_data), "Val:", len(val_data))

# --- 3. Bigram Model ---
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --- 4. Training ---
torch.manual_seed(1337)
batch_size = 32
block_size = 8
model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

print("\n--- Training Start ---")
for step in range(300):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

print("Final Loss:", loss.item())

# --- 5. Generate ---
print("\n--- Generated Text (Bigram) ---")
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=300)[0].tolist()
print(decode(generated))

Total characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
Vocab size: 65
Train: 1003854 Val: 111540

--- Training Start ---
Step 0 | Loss: 4.6485
Step 50 | Loss: 4.5812
Step 100 | Loss: 4.6430
Step 150 | Loss: 4.4494
Step 200 | Loss: 4.4735
Step 250 | Loss: 4.4248
Final Loss: 4.378273010253906

--- Generated Text (Bigram) ---

p:Cxfxlv fgiObE!$ !qYKottToEHLzMBTsvvXRv!YG:wi-w'niMV&$-MKu;Sb
lajuDClaov&jy$vhscnhSlsa!iHDPs!w'QCVuDGybsqV&dF-mZYDugCV3SxGJpxE!vimR
SVbBZRCdmsiPxrdDPTEJ:CsBfaooXeLA-!UsBGQTReO q-xTkgw'y kSKxc;AcLJfqM:bjxZGmrD'lnxErV&yzq,Kp?dWnq;A:3Q$JrdsgXk$Uis,rkzGSjKoisiWu-k&M;agYWrwbTq- Km$vzIVBT'MV&
SP&D3h$gq q


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Hyperparameters (CPU friendly) ---
batch_size = 32
block_size = 64
max_iters = 1000
eval_interval = 200
learning_rate = 3e-4
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.0
torch.manual_seed(1337)

# --- Data (wahi purana) ---
with open('/content/drive/MyDrive/tiny-gpt-from-scratch/data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_interval)
        for k in range(eval_interval):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# --- Transformer Blocks ---
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel()
print("Total Params:", sum(p.numel() for p in model.parameters())/1e6, "M")
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# --- Training ---
print("Training Start... CPU pe 1000 steps me 15-25 min lag sakte hai, ghabrana mat")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# --- Save + Generate ---
torch.save(model.state_dict(), '/content/drive/MyDrive/tiny-gpt-from-scratch/checkpoints/gpt_model.pt')
print("Model Saved!")

print("\n--- Generated Text (GPT) ---")
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=500)[0].tolist()
gen_text = decode(generated)
print(gen_text)

with open('/content/drive/MyDrive/tiny-gpt-from-scratch/samples_sample.txt', 'w') as f:
    f.write(gen_text)
print("\nSample saved!")

Total Params: 0.816705 M
Training Start... CPU pe 1000 steps me 15-25 min lag sakte hai, ghabrana mat
step 0: train loss 4.3354, val loss 4.3307
step 200: train loss 2.5098, val loss 2.5100
step 400: train loss 2.3809, val loss 2.3985
step 600: train loss 2.2668, val loss 2.2836
step 800: train loss 2.1688, val loss 2.1962
step 999: train loss 2.0851, val loss 2.1234
Model Saved!

--- Generated Text (GPT) ---

Ne? Ofor seer hean.

MENVOR:
I no ewfll I pecostser.

Noserds wisgn, tharde wevan me tly wontrsp, of reshiss pen apst;
s of rante, Ike in pus gine the.

LORY I I Ifrat shor thou lit hou,
Gome, him tee wif chees thing a ingrake!

If qu:
She worte forge fary Gotyd vother wheloul am
The theon ealbavin becess, afuisee! and, in shop,
The to in of thene bur cathemegh,
I Rher asall prisiest in shax ingeay:
ca nof murd, whall by shall hith yourNoot himell:
Thes do thing for jon mir by ase?

MARENO:
So s

Sample saved!
